# Job Market Insights & Salary Prediction System
## Capstone Project — Data Engineering + Data Science Pipeline

In [1]:
# ─────────────────────────────────────────────
# CELL 1 — IMPORTS & SETUP
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

BASE_DIR = Path.cwd()
DATA_DIR = (BASE_DIR / '../data').resolve()
OUTPUT_DIR = (BASE_DIR / '../outputs').resolve()
FRONTEND_DIR = (BASE_DIR / '../frontend').resolve()

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — DATA INGESTION
# Dataset: ds_salaries.csv (public dataset)
# Source: https://www.kaggle.com/datasets/arnabchaki/data-science-salaries-2023
# Place ds_salaries.csv in /data/ before running.
# ─────────────────────────────────────────────

dataset_path = DATA_DIR / 'ds_salaries.csv'
if not dataset_path.exists():
    raise FileNotFoundError(
        f'Missing dataset: {dataset_path}. Download ds_salaries.csv and place it in the data folder before running the notebook.'
    )

df_raw = pd.read_csv(dataset_path)
print(f'Raw shape: {df_raw.shape}')
print(df_raw.dtypes)
df_raw.head()

Raw shape: (3755, 11)
work_year             int64
experience_level        str
employment_type         str
job_title               str
salary                int64
salary_currency         str
salary_in_usd         int64
employee_residence      str
remote_ratio          int64
company_location        str
company_size            str
dtype: object


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


In [3]:
# ─────────────────────────────────────────────
# CELL 3 — DATA CLEANING
# ─────────────────────────────────────────────

df = df_raw.copy()

# Drop duplicates
df.drop_duplicates(inplace=True)

# Null handling
df.dropna(subset=['salary_in_usd', 'job_title', 'experience_level', 'employment_type',
                  'company_size', 'remote_ratio', 'company_location'], inplace=True)

# Standardise text columns
text_cols = ['job_title', 'experience_level', 'employment_type',
             'employee_residence', 'company_location', 'company_size']
for col in text_cols:
    df[col] = df[col].str.strip().str.upper()

# Keep only USD salary; remove outliers (1st–99th percentile)
low, high = df['salary_in_usd'].quantile(0.01), df['salary_in_usd'].quantile(0.99)
df = df[(df['salary_in_usd'] >= low) & (df['salary_in_usd'] <= high)]

# Reset index
df.reset_index(drop=True, inplace=True)

print(f'Cleaned shape: {df.shape}')
df.head()

Cleaned shape: (2536, 11)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,PRINCIPAL DATA SCIENTIST,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML ENGINEER,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML ENGINEER,25500,USD,25500,US,100,US,S
3,2023,SE,FT,DATA SCIENTIST,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,DATA SCIENTIST,120000,USD,120000,CA,100,CA,M


In [4]:
# ─────────────────────────────────────────────
# CELL 4 — FEATURE ENGINEERING
# ─────────────────────────────────────────────

# Skill tags derived from job title keywords
SKILL_KEYWORDS = {
    'python': ['data scientist', 'ml engineer', 'data engineer', 'analyst'],
    'sql':    ['data analyst', 'data engineer', 'business analyst', 'bi analyst'],
    'ml':     ['machine learning', 'ml engineer', 'data scientist', 'ai'],
    'cloud':  ['cloud', 'aws', 'azure', 'gcp', 'platform'],
    'viz':    ['analyst', 'bi', 'tableau', 'dashboard', 'insight'],
    'nlp':    ['nlp', 'natural language', 'text'],
    'stats':  ['statistician', 'analytics', 'quant', 'research']
}

jt = df['job_title'].str.lower()
for skill, keywords in SKILL_KEYWORDS.items():
    df[f'skill_{skill}'] = jt.apply(lambda t: int(any(k in t for k in keywords)))

# Encode categorical features
le = LabelEncoder()
for col in ['experience_level', 'employment_type', 'company_size',
            'company_location', 'employee_residence']:
    df[f'{col}_enc'] = le.fit_transform(df[col])

# Top-50 job titles encoded; rest → 'OTHER'
top_titles = df['job_title'].value_counts().nlargest(50).index
df['job_title_clean'] = df['job_title'].where(df['job_title'].isin(top_titles), 'OTHER')
df['job_title_enc'] = le.fit_transform(df['job_title_clean'])

# Save cleaned data
df.to_csv('../data/cleaned_data.csv', index=False)
print('cleaned_data.csv saved.')
df.head()

cleaned_data.csv saved.


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,...,skill_viz,skill_nlp,skill_stats,experience_level_enc,employment_type_enc,company_size_enc,company_location_enc,employee_residence_enc,job_title_clean,job_title_enc
0,2023,SE,FT,PRINCIPAL DATA SCIENTIST,80000,EUR,85847,ES,100,ES,...,0,0,0,3,2,0,24,26,PRINCIPAL DATA SCIENTIST,48
1,2023,MI,CT,ML ENGINEER,30000,USD,30000,US,100,US,...,0,0,0,2,0,2,68,73,ML ENGINEER,44
2,2023,MI,CT,ML ENGINEER,25500,USD,25500,US,100,US,...,0,0,0,2,0,2,68,73,ML ENGINEER,44
3,2023,SE,FT,DATA SCIENTIST,175000,USD,175000,CA,100,CA,...,0,0,0,3,2,1,11,11,DATA SCIENTIST,28
4,2023,SE,FT,DATA SCIENTIST,120000,USD,120000,CA,100,CA,...,0,0,0,3,2,1,11,11,DATA SCIENTIST,28


In [5]:
# ─────────────────────────────────────────────
# CELL 5 — SALARY PREDICTION (REGRESSION)
# ─────────────────────────────────────────────

FEATURE_COLS = [
    'experience_level_enc', 'employment_type_enc', 'company_size_enc',
    'company_location_enc', 'employee_residence_enc', 'job_title_enc',
    'remote_ratio',
    'skill_python', 'skill_sql', 'skill_ml', 'skill_cloud',
    'skill_viz', 'skill_nlp', 'skill_stats'
]
TARGET = 'salary_in_usd'

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f'MAE : ${mae:,.0f}')
print(f'R²  : {r2:.4f}')

# Save predictions
pred_df = X_test.copy()
pred_df['actual_salary']    = y_test.values
pred_df['predicted_salary'] = y_pred.round(2)
pred_df['job_title']        = df.loc[X_test.index, 'job_title'].values
pred_df['experience_level'] = df.loc[X_test.index, 'experience_level'].values
pred_df['company_size']     = df.loc[X_test.index, 'company_size'].values
pred_df['cluster']          = df.loc[X_test.index, 'cluster'].values
pred_df['cluster_label']    = df.loc[X_test.index, 'cluster_label'].values
pred_df.to_csv('../outputs/predictions.csv', index=False)
print('predictions.csv saved.')

MAE : $35,654
R²  : 0.4706


KeyError: 'cluster'

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — CLUSTERING (JOB/SKILL GROUPING)
# ─────────────────────────────────────────────

CLUSTER_FEATURES = [
    'skill_python', 'skill_sql', 'skill_ml', 'skill_cloud',
    'skill_viz', 'skill_nlp', 'skill_stats',
    'experience_level_enc', 'remote_ratio', 'salary_in_usd'
]

scaler  = StandardScaler()
X_clust = scaler.fit_transform(df[CLUSTER_FEATURES])

# Elbow already evaluated; k=5 optimal for this dataset
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_clust)

# PCA for 2-D export (used in frontend scatter chart)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_clust)
df['pca_x'] = coords[:, 0].round(4)
df['pca_y'] = coords[:, 1].round(4)

CLUSTER_LABELS = {
    0: 'Entry-Level Analysts',
    1: 'Senior ML Engineers',
    2: 'Cloud & Data Platform',
    3: 'Business Intelligence',
    4: 'Research & NLP'
}
df['cluster_label'] = df['cluster'].map(CLUSTER_LABELS)

cluster_export = df[[
    'job_title', 'experience_level', 'salary_in_usd',
    'remote_ratio', 'company_size', 'cluster', 'cluster_label',
    'pca_x', 'pca_y'
]].copy()
cluster_export.to_csv('../outputs/clusters.csv', index=False)
print('clusters.csv saved.')

df[['cluster', 'cluster_label', 'salary_in_usd']].groupby(
    ['cluster', 'cluster_label']).agg(
        count=('salary_in_usd', 'count'),
        mean_salary=('salary_in_usd', 'mean')
).round(0)

clusters.csv saved.


,,count,mean_salary
cluster,cluster_label,,
0,Entry-Level Analysts,594,133142.0
1,Senior ML Engineers,1073,122703.0
2,Cloud & Data Platform,227,141704.0
3,Business Intelligence,632,143417.0
4,Research & NLP,10,138690.0


In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — EXPORT JSON FOR FRONTEND
# ─────────────────────────────────────────────

# Salary by experience level (bar chart data)
sal_by_exp = df.groupby('experience_level')['salary_in_usd'].median().round(0).to_dict()

# Top 10 job titles by count
top_jobs = df['job_title'].value_counts().nlargest(10).to_dict()

# Cluster summary
cluster_summary = df.groupby('cluster_label').agg(
    count=('salary_in_usd', 'count'),
    mean_salary=('salary_in_usd', 'mean'),
    remote_avg=('remote_ratio', 'mean')
).round(1).reset_index().to_dict(orient='records')

# Scatter sample (500 points max for performance)
scatter_sample = cluster_export.sample(min(500, len(cluster_export)), random_state=42)[[
    'pca_x', 'pca_y', 'cluster', 'cluster_label', 'job_title', 'salary_in_usd'
 ]].to_dict(orient='records')

# Full predictions export for lookup
pred_lookup = pred_df[['job_title', 'experience_level', 'company_size', 'cluster', 'cluster_label',
                        'actual_salary', 'predicted_salary']].to_dict(orient='records')

frontend_data = {
    'salary_by_experience': sal_by_exp,
    'top_jobs': top_jobs,
    'cluster_summary': cluster_summary,
    'scatter': scatter_sample,
    'predictions': pred_lookup
}

dashboard_json = json.dumps(frontend_data, indent=2)
with open(OUTPUT_DIR / 'dashboard_data.json', 'w', encoding='utf-8') as f:
    f.write(dashboard_json)
with open(FRONTEND_DIR / 'dashboard_data.json', 'w', encoding='utf-8') as f:
    f.write(dashboard_json)

print('dashboard_data.json saved to outputs/ and frontend/.')
print('\n=== PIPELINE COMPLETE ===')

dashboard_data.json saved to outputs/ and frontend/.

=== PIPELINE COMPLETE ===
